# Exercício 8 — Caminho Mínimo em um Dígrafo

Dado um dígrafo com localidades como vértices e distâncias nos arcos, determinar
o **caminho de menor custo** entre um vértice de origem e um vértice destino.

**Dígrafo do exercício** (distâncias em cada arco):

| Arco (i → j) | Distância |
|:---:|:---:|
| A → B | 6 |
| A → C | 7 |
| B → C | 1 |
| B → D | 3 |
| B → E | 4 |
| C → D | 2 |
| C → F | 5 |
| D → E | 3 |
| D → F | 2 |
| E → G | 7 |
| F → E | 2 |
| F → G | 4 |

**Origem:** A &nbsp;&nbsp; **Destino:** G

---

## Formulação — Fluxo em Rede (ILP)

Variáveis de decisão binárias $x_{ij} \in \{0,1\}$: vale 1 se o arco $(i,j)$ pertence ao caminho mínimo.

$$\min \sum_{(i,j) \in A} c_{ij}\, x_{ij}$$

Sujeito à **conservação de fluxo** em cada vértice $v$:

$$\sum_{j:(v,j)\in A} x_{vj} - \sum_{i:(i,v)\in A} x_{iv} =
\begin{cases}
 1  & \text{se } v = \text{origem} \\
-1  & \text{se } v = \text{destino} \\
 0  & \text{caso contrário}
\end{cases}$$

$$x_{ij} \in \{0, 1\}, \quad \forall (i,j) \in A$$

In [1]:
!pip install ortools -q

In [2]:
# ── ENTRADAS GENÉRICAS ────────────────────────────────────────────────────────

vertices = ['A', 'B', 'C', 'D', 'E', 'F', 'G']

# Lista de arcos: (origem, destino, peso)
arcos = [
    ('A', 'B', 6),
    ('A', 'C', 7),
    ('B', 'C', 1),
    ('B', 'D', 3),
    ('B', 'E', 4),
    ('C', 'D', 2),
    ('C', 'F', 5),
    ('D', 'E', 3),
    ('D', 'F', 2),
    ('E', 'G', 7),
    ('F', 'E', 2),
    ('F', 'G', 4),
]

origem  = 'A'
destino = 'G'
# ─────────────────────────────────────────────────────────────────────────────

In [3]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver('SCIP')
if solver is None:
    raise RuntimeError('Não foi possível criar o solver SCIP.')

# Variáveis binárias: x[(i,j)] = 1 se o arco (i→j) está no caminho
x = {(i, j): solver.BoolVar(f'x_{i}_{j}') for i, j, _ in arcos}

In [4]:
# Função objetivo: minimizar o custo total do caminho
objetivo = solver.Objective()
for i, j, c in arcos:
    objetivo.SetCoefficient(x[(i, j)], c)
objetivo.SetMinimization()

In [5]:
# Restrições de conservação de fluxo em cada vértice
for v in vertices:
    if v == origem:
        b = 1
    elif v == destino:
        b = -1
    else:
        b = 0

    saida  = [x[(i, j)] for i, j, _ in arcos if i == v]
    entrada = [x[(i, j)] for i, j, _ in arcos if j == v]

    ct = solver.RowConstraint(b, b, f'fluxo_{v}')
    for var in saida:
        ct.SetCoefficient(var,  1)
    for var in entrada:
        ct.SetCoefficient(var, -1)

In [6]:
print(solver.ExportModelAsLpFormat(False))

\ Generated by MPModelProtoExporter
\   Name             : 
\   Format           : Free
\   Constraints      : 7
\   Variables        : 12
\     Binary         : 12
\     Integer        : 0
\     Continuous     : 0
Minimize
 Obj: +6 x_A_B +7 x_A_C +1 x_B_C +3 x_B_D +4 x_B_E +2 x_C_D +5 x_C_F +3 x_D_E +2 x_D_F +7 x_E_G +2 x_F_E +4 x_F_G 
Subject to
 fluxo_A: +1 x_A_B +1 x_A_C  = 1
 fluxo_B: -1 x_A_B +1 x_B_C +1 x_B_D +1 x_B_E  = 0
 fluxo_C: -1 x_A_C -1 x_B_C +1 x_C_D +1 x_C_F  = 0
 fluxo_D: -1 x_B_D -1 x_C_D +1 x_D_E +1 x_D_F  = 0
 fluxo_E: -1 x_B_E -1 x_D_E +1 x_E_G -1 x_F_E  = 0
 fluxo_F: -1 x_C_F -1 x_D_F +1 x_F_E +1 x_F_G  = 0
 fluxo_G: -1 x_E_G -1 x_F_G  = -1
Bounds
 0 <= x_A_B <= 1
 0 <= x_A_C <= 1
 0 <= x_B_C <= 1
 0 <= x_B_D <= 1
 0 <= x_B_E <= 1
 0 <= x_C_D <= 1
 0 <= x_C_F <= 1
 0 <= x_D_E <= 1
 0 <= x_D_F <= 1
 0 <= x_E_G <= 1
 0 <= x_F_E <= 1
 0 <= x_F_G <= 1
Binaries
 x_A_B
 x_A_C
 x_B_C
 x_B_D
 x_B_E
 x_C_D
 x_C_F
 x_D_E
 x_D_F
 x_E_G
 x_F_E
 x_F_G
End



In [7]:
status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    custo_minimo = objetivo.Value()
    arcos_no_caminho = [(i, j, c) for i, j, c in arcos if x[(i, j)].solution_value() > 0.5]

    # Reconstrói a sequência de vértices do caminho
    adj = {i: j for i, j, _ in arcos_no_caminho}
    caminho = [origem]
    atual = origem
    while atual != destino:
        atual = adj[atual]
        caminho.append(atual)

    print(f'Custo mínimo de {origem} até {destino}: {int(custo_minimo)}')
    print(f'Caminho: {" → ".join(caminho)}')
    print()
    print('Arcos utilizados:')
    for i, j, c in arcos_no_caminho:
        print(f'  {i} → {j}  (distância {c})')
else:
    print('Modelo sem solução ótima.')

Custo mínimo de A até G: 15
Caminho: A → C → D → F → G

Arcos utilizados:
  A → C  (distância 7)
  C → D  (distância 2)
  D → F  (distância 2)
  F → G  (distância 4)
